# Software-maintainability baseline models

Run this notebook from the repository root after following the README setup. Select a dataset, run all cells, and compare the five models. Source CSV files are in `data/`; the results CSV is saved in `outputs/`.

In [ ]:
# Uncomment and run each line once if the package is missing.
# %pip install jupyterlab
# %pip install numpy
# %pip install pandas
# %pip install scikit-learn

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor

DATASET = "all"  # Choose: "all", "QUES", or "UIMS"
OUTPUT_FILE = Path("outputs/baseline_results.csv")
DATASETS = {"QUES": Path("data/QUES.csv"), "UIMS": Path("data/UIMS1.csv")}
FEATURES = ("dit", "noc", "mpc", "rfc", "lcom", "dac", "wmc", "nom", "size2", "size1")
TARGET = "change"


def models():
    return {
        "Linear regression": LinearRegression(),
        "Ridge regression": make_pipeline(StandardScaler(), Ridge(alpha=1.0)),
        "K-nearest neighbors": make_pipeline(StandardScaler(), KNeighborsRegressor(n_neighbors=5)),
        "Decision tree": DecisionTreeRegressor(random_state=0),
        "Support vector regression": make_pipeline(StandardScaler(), SVR(kernel="rbf")),
    }


def evaluate_dataset(name):
    dataframe = pd.read_csv(DATASETS[name])
    features = dataframe.loc[:, list(FEATURES)]
    target = dataframe[TARGET]
    train_features, test_features, train_target, test_target = train_test_split(
        features, target, test_size=0.2, random_state=0
    )

    rows = []
    for model_name, model in models().items():
        model.fit(train_features, train_target)
        prediction = model.predict(test_features)
        rows.append({
            "Dataset": name,
            "Model": model_name,
            "R2": r2_score(test_target, prediction),
            "MAE": mean_absolute_error(test_target, prediction),
            "RMSE": np.sqrt(mean_squared_error(test_target, prediction)),
            "Train samples": len(train_target),
            "Test samples": len(test_target),
        })
    return pd.DataFrame(rows).sort_values("RMSE", ignore_index=True)


def evaluate(dataset):
    dataset_names = DATASETS if dataset == "all" else (dataset,)
    return pd.concat([evaluate_dataset(name) for name in dataset_names], ignore_index=True)


In [ ]:
results = evaluate(DATASET)
OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
results.to_csv(OUTPUT_FILE, index=False)

print(f"Saved results to {OUTPUT_FILE}")
results